In [1]:
# Load Data

import pandas as pd

file_path = "C:\\Users\\Admin\\Downloads\\Portfolio Projects\\E-Commerce-RFM-Analytics\\data\\online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

print("Available Worksheets:")
print(excel_file.sheet_names)

sheet1 = excel_file.parse("Year 2009-2010")
sheet2 = excel_file.parse("Year 2010-2011")

sheets = {
    "2009-2010": sheet1,
    "2010-2011": sheet2
}

Available Worksheets:
['Year 2009-2010', 'Year 2010-2011']


In [2]:
# 1. Combine Transaction Sheets

df = pd.concat([sheet1, sheet2], ignore_index = True)

print("Combined dataset shape : ", df.shape)

Combined dataset shape :  (1067371, 8)


In [3]:
# 2. Creating Transaction Flags

df["Is Cancellation"] = df["Invoice"].str.startswith("C", na = False)

df["Is Return"] = df["Quantity"] < 0

df["Is Zero Price"] = df["Price"] == 0

df["Is Manual"] = df["StockCode"] == "M"

df["Is Test"] = df["StockCode"] == "TEST001"

df["Is Bad Debt"] = (
    (df["StockCode"] == "B") & 
    (df["Description"] == "Adjust bad debt") 
)

df["Is Exact Duplicate"] = df.duplicated()

In [4]:
# 3. Remove Non-Product / Accounting Records

before = len(df)

df_clean = df[~(df["Is Manual"] | df["Is Test"] | df["Is Bad Debt"]).copy()]

after = len(df_clean)

print("\n--- Non-Product Removal ---")
print("Rows before removal : ", before)
print("Rows removed : ", before - after)
print("Rows remaining : ", after)


--- Non-Product Removal ---
Rows before removal :  1067371
Rows removed :  1442
Rows remaining :  1065929


In [5]:
# 4. Remove exact duplicate records

before = len(df_clean)

df_clean = df_clean[~df_clean["Is Exact Duplicate"]].copy()

after = len(df_clean)

print("\n--- Exact Duplicate Removal ---")
print("Rows before removal : ", before)
print("Duplicate rows removed : ", before - after)
print("Rows remaining : ", after)


--- Exact Duplicate Removal ---
Rows before removal :  1065929
Duplicate rows removed :  34301
Rows remaining :  1031628


In [6]:
# 5. Remove Transactions with Missing Customer ID

before = len(df_clean)

df_clean = df_clean[df_clean["Customer ID"].notna()].copy()

after = len(df_clean)

print("\n--- Missing Customer ID Removal ---")
print("Rows before removal : ", before)
print("Rows removed : ", before - after)
print("Rows remaining : ", after)


--- Missing Customer ID Removal ---
Rows before removal :  1031628
Rows removed :  234843
Rows remaining :  796785


In [7]:
# 6. Remove Negative-Quantity Keep Positive-Quantity Sales Transactions

before = len(df_clean)

df_clean = df_clean[~df_clean["Is Return"]].copy()

after = len(df_clean)

print("\n--- Return / Cancellation Removal ---")
print("Rows before removal : ", before)
print("Return/Cancellation rows removed : ", before - after)
print("Rows remaining : ", after)


--- Return / Cancellation Removal ---
Rows before removal :  796785
Return/Cancellation rows removed :  17989
Rows remaining :  778796


In [8]:
# 7. Remove Zero-Price Transactions

before = len(df_clean)

df_clean = df_clean[~df_clean["Is Zero Price"]].copy()

after = len(df_clean)

print("\n--- Zero-Price Removal ---")
print("Rows before  removal : ", before)
print("Zero-Price rows removed : ", before - after)
print("Rows remaining : ", after)


--- Zero-Price Removal ---
Rows before  removal :  778796
Zero-Price rows removed :  61
Rows remaining :  778735


In [9]:
# 8. Calculate Transaction Revenue

df_clean["Total Amount"] = (df_clean["Quantity"] * df_clean["Price"]).round(4)

print("\n--- Transaction Revenue ---")
print(df_clean[["Invoice", "StockCode", 
                "Quantity", "Price", "Total Amount"]].head())


--- Transaction Revenue ---
  Invoice StockCode  Quantity  Price  Total Amount
0  489434     85048        12   6.95          83.4
1  489434    79323P        12   6.75          81.0
2  489434    79323W        12   6.75          81.0
3  489434     22041        48   2.10         100.8
4  489434     21232        24   1.25          30.0


In [10]:
# 9. Final Cleaning Validation

print("\n" + "_" * 60)
print("\nFINAL CLEANING VALIDATION")
print("_" * 60)

print("\nFinal Shape : ", df_clean.shape)

print("\nMissing Customer ID : ", df_clean["Customer ID"].isna().sum())

print("\nZero-Quantity : ", (df_clean["Quantity"] == 0).sum())

print("\nNegative Quantity : ", (df_clean["Quantity"] < 0).sum())

print("\nManual Records : ", df_clean["Is Manual"].sum())

print("\nTEST001 Records : ", df_clean["Is Test"].sum())

print("\nBad Debt Records : ", df_clean["Is Bad Debt"].sum())

print("\nExact Duplicates : ", df_clean.duplicated().sum())

print("\nTotal Amount Summary : ", df_clean["Total Amount"].describe())


____________________________________________________________

FINAL CLEANING VALIDATION
____________________________________________________________

Final Shape :  (778735, 16)

Missing Customer ID :  0

Zero-Quantity :  0

Negative Quantity :  0

Manual Records :  0

TEST001 Records :  0

Bad Debt Records :  0

Exact Duplicates :  0

Total Amount Summary :  count    778735.000000
mean         22.116383
std         225.875405
min           0.001000
25%           4.950000
50%          12.480000
75%          19.800000
max      168469.600000
Name: Total Amount, dtype: float64


In [ ]:
# 10. Select clean columns and Export Dataset

clean_columns = ["Invoice", "StockCode", 
                 "Description", "Quantity", 
                 "InvoiceDate", "Price", 
                 "Customer ID", "Country", 
                 "Total Amount"]

output_path = "C:\\Users\\Admin\\Downloads\\Portfolio Projects\\E-Commerce-RFM-Analytics\\data\\online_retail_cleaned.csv"

# Export only essential production columns with explicit line endings
df_clean[clean_columns].to_csv(
    output_path, index = False, 
    lineterminator = '\n')

print("\nCleaned production dataset saved Successfully!")
print("File : ", output_path)